# 02 · Surrogate checks, PGD baseline and edit check (the go/no-go gate)

Every code cell sets itself up, so any one can be run alone on a fresh runtime.

1. Check that every surrogate behaves on clean images (identity matches its
   mirror, CLIP is stable under blur, VAE latents are sane). A FAIL here means
   a preprocessing bug: fix it before anything else.
2. Run per-image PGD+EOT at 8/255 and 16/255 on 200 held-out images.

**Gate:** after `jpeg75`, the PGD rows should show clearly lower `*_cos` and
`*_ratio` than the `clean` rows. If 8/255 barely moves them, fix the loss or
the augmentations before training the generator.

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=True, gpu=True, requirements=('requirements.txt',))
# -----------------------------------------------------------------------------

!python scripts/check_surrogates.py --config configs/pgd.yaml --out {DRIVE_ROOT}/surrogate_check
from IPython.display import Image as Show
Show(f"{DRIVE_ROOT}/surrogate_check/aligned_faces.png")

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=True, gpu=True, requirements=('requirements.txt',))
# -----------------------------------------------------------------------------

!python -m soteria.pgd --config configs/pgd.yaml eps=0.03137 out_dir={DRIVE_ROOT}/pgd/eps8

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=True, gpu=True, requirements=('requirements.txt',))
# -----------------------------------------------------------------------------

!python -m soteria.pgd --config configs/pgd.yaml eps=0.06275 out_dir={DRIVE_ROOT}/pgd/eps16

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=False, gpu=False, requirements=())
# -----------------------------------------------------------------------------

import json, pandas as pd
for run in ("eps8", "eps16"):
    s = json.load(open(f"{DRIVE_ROOT}/pgd/{run}/summary.json"))
    print(f"== {run}: {s['sec_per_image']:.1f} s/image")
    display(pd.DataFrame(s["by_method"]).round(4))

## Ablation: is the VAE term starved, or just hard? (8/255, 64 images, ~25 min each)

In the baseline, CLIP and face are broken at 8/255 but the VAE ratio only
falls 0.985 → 0.897 after JPEG.

- **vae_only** turns the CLIP and face terms off. It is the most the VAE can
  move at 8/255 under this EOT. If it also stalls near 0.85, the VAE
  objective itself is the problem, not the balance between terms.
- **rebalanced** keeps all three, but stops the CLIP / face terms pulling once
  an image is past an unrelated photo / a stranger (`clip_floor=0.3`,
  `face_floor=-0.1`, from the surrogate check: 0.425 and -0.002), and gives
  the VAE 4x weight.

The 64 images are the first of the 200 above, so the last cell compares
all three on exactly the same images.

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=True, gpu=True, requirements=('requirements.txt',))
# -----------------------------------------------------------------------------

!python -m soteria.pgd --config configs/pgd.yaml eps=0.03137 pgd_images=64 w_clip=0 w_face=0 out_dir={DRIVE_ROOT}/pgd/abl_vae_only

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=True, gpu=True, requirements=('requirements.txt',))
# -----------------------------------------------------------------------------

!python -m soteria.pgd --config configs/pgd.yaml eps=0.03137 pgd_images=64 clip_floor=0.3 face_floor=-0.1 w_vae=4 out_dir={DRIVE_ROOT}/pgd/abl_rebalanced

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=False, gpu=False, requirements=())
# -----------------------------------------------------------------------------

import pandas as pd
runs = {"baseline": "eps8", "vae_only": "abl_vae_only", "rebalanced": "abl_rebalanced"}
dfs = {name: pd.read_csv(f"{DRIVE_ROOT}/pgd/{d}/metrics.csv") for name, d in runs.items()}
same = set(dfs["vae_only"]["index"])
cols = [c for c in dfs["baseline"].columns if c.startswith("jpeg75/") and not c.endswith("_shift")]
rows = {"clean": dfs["baseline"].query("method == 'clean'")}
rows.update({name: df.query("method == 'pgd'") for name, df in dfs.items()})
table = pd.DataFrame({name: df[df["index"].isin(same)][cols + ["lpips", "psnr"]].mean() for name, df in rows.items()})
print(f"{len(same)} images, after a real JPEG q75 round trip (lower = better protection)")
display(table.round(4))

## Edit check: does a cloak actually spoil a Stable Diffusion edit? (~30 min)

The VAE ratio is only a proxy. This runs SD1.5 img2img (strength 0.5) on 16 of
the images above (12 faces, 4 scenes): the clean photo, two controls and every
PGD run, all with the same prompt and the same seeded noise. It runs at 256 px
(the size the cloaks were made at) and 512 px (SD1.5's native size, the cloak
upscaled).

Controls:
- `clean_again`: the clean photo edited a second time. Its score is the
  noise floor of the GPU itself; anything near it means no effect.
- `random8`: random noise at the full 8/255 budget.

Read the results like this:
- `lpips_vs_clean_edit`: how different the edit of the cloaked photo is from
  the edit of the clean one. A cloak works only if this is **clearly above
  `random8`**, not just above 0.
- `id_cos`: ArcFace similarity of the edit to the original person. The
  `clean` row is what the edit keeps by itself; lower than that = identity lost.
- `grid_*.png`: rows are images; columns are the clean input, then the edit of
  clean, clean_again, random8, pgd8, vae_only, rebalanced, pgd16. Look for edits that are
  visibly wrecked (smears, grey patches, wrong content), not just different.

Writes `{DRIVE_ROOT}/edit_check/`: grid_256.png, grid_512.png, metrics.csv, summary.csv.

In [ ]:
# ---- setup: lets this cell run on its own; safe to rerun --------------------
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # data tar, runs and results live here
import importlib, os, subprocess, sys
from google.colab import drive, userdata
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
CODE = "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    try:
        _url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:          # no GITHUB_TOKEN secret: fine for a public repo
        _url = REPO_URL
    _git("clone", "-q", "-b", BRANCH, _url, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.colab; importlib.reload(soteria.colab)
soteria.colab.prepare(DRIVE_ROOT, data=True, gpu=True, requirements=('requirements.txt',))
# -----------------------------------------------------------------------------

!python -m soteria.eval.edit --pgd-root {DRIVE_ROOT}/pgd --out {DRIVE_ROOT}/edit_check
from IPython.display import Image as Show, display
for size in (256, 512):
    print(f"grid {size}px: clean input | edit of clean, clean_again, random8, pgd8, vae_only, rebalanced, pgd16")
    display(Show(f"{DRIVE_ROOT}/edit_check/grid_{size}.png", width=900))